hello


## here is the just simple code tried by sameer and just 

In [2]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [7]:
train_data = datasets.ImageFolder("data/processed/train", transform=train_transform)
val_data = datasets.ImageFolder("data/processed/val", transform=val_test_transform)
test_data = datasets.ImageFolder("data/processed/test", transform=val_test_transform)

In [8]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=False)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [17]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            predicted = (outputs > 0.5).float()

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [9]:
import torch
import torch.nn as nn

class DenseNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*128*3, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = DenseNN()

In [10]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [18]:
def train_model(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}")

In [19]:
train_model(model, train_loader, val_loader, epochs=5)

Epoch 1, Loss: 4.3588, Val Acc: 0.8000
Epoch 2, Loss: 4.3610, Val Acc: 0.8000
Epoch 3, Loss: 4.3539, Val Acc: 0.8000
Epoch 4, Loss: 4.3430, Val Acc: 0.8000
Epoch 5, Loss: 4.3329, Val Acc: 0.8000


In [21]:
from collections import Counter
print(Counter([label for _, label in train_data]))

Counter({1: 289, 0: 46})


In [13]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),   # (128 → 126)
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (126 → 63)

            nn.Conv2d(32, 64, kernel_size=3),  # (63 → 61)
            nn.ReLU(),
            nn.MaxPool2d(2),                   # (61 → 30)

            nn.Conv2d(64, 128, kernel_size=3), # (30 → 28)
            nn.ReLU(),
            nn.MaxPool2d(2)                    # (28 → 14)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 14 * 14, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [14]:
cnn_model = CNN()

In [15]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

In [20]:
train_model(cnn_model, train_loader, val_loader, epochs=5)

Epoch 1, Loss: 4.4081, Val Acc: 0.8133
Epoch 2, Loss: 4.1827, Val Acc: 0.8133
Epoch 3, Loss: 4.2594, Val Acc: 0.8133
Epoch 4, Loss: 3.8923, Val Acc: 0.8133
Epoch 5, Loss: 3.6834, Val Acc: 0.8133


In [22]:
check_predictions(cnn_model, val_loader)

NameError: name 'check_predictions' is not defined

In [23]:
class_weights = torch.tensor([289/46])  # weight for class 0
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)

In [24]:
class_weights

tensor([6.2826])